# REPA × CheXpert × MedDINOv3 (Colab)

Trains **SiT-S/8** on CheXpert chest X-rays using **MedDINOv3 ViT-Base** as the representation encoder and **MedVAE** as the diffusion VAE.

**Requirements:**
- Kaggle API credentials (`kaggle.json`) — used to download CheXpert and MedDINOv3
- Google Drive — results and checkpoints are saved there to survive session resets

**Runtime:** GPU (T4 or better)

## 1. Install dependencies

In [ ]:
!pip install -q accelerate diffusers timm einops pandas medvae kaggle

## 2. Mount Google Drive

Checkpoints and results are saved to Drive so they persist across session resets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
RESULTS_DIR = '/content/drive/MyDrive/repa-results'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Drive mounted. Results will be saved to:', RESULTS_DIR)

## 3. Setup Kaggle credentials & download datasets

Upload your `kaggle.json` when prompted, then the datasets will be downloaded automatically.

In [ ]:
import os
from google.colab import files

# Upload kaggle.json
print('Upload your kaggle.json:')
uploaded = files.upload()
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
os.rename('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
print('Kaggle credentials set.')

In [ ]:
import subprocess

# Download CheXpert
if not os.path.isfile('/content/chexpert/train.csv'):
    print('Downloading CheXpert...')
    subprocess.run(['kaggle', 'datasets', 'download', 'ashery/chexpert',
                    '-p', '/content/chexpert', '--unzip'], check=True)
else:
    print('CheXpert already downloaded.')

# Download MedDINOv3
if not os.path.isdir('/content/meddinov3/meddinov3_minimal'):
    print('Downloading MedDINOv3...')
    subprocess.run(['kaggle', 'datasets', 'download', 'nikiboura/meddinov3-minimal',
                    '-p', '/content/meddinov3', '--unzip'], check=True)
else:
    print('MedDINOv3 already downloaded.')

print('Done.')

## 4. Clone REPA fork

In [ ]:
%%bash
cd /content
if [ ! -d "REPA" ]; then
    git clone https://github.com/nikiboura/REPA.git REPA
fi
cd REPA && git pull
echo "REPA ready."

## 5. Set paths

In [ ]:
import os

REPA_DIR      = '/content/REPA'
DATA_DIR      = '/content/data/chexpert_256'
CHEXPERT_ROOT = '/content/chexpert'
MEDDINOV3     = '/content/meddinov3/meddinov3_minimal'
CKPT_PATH     = '/content/meddinov3/meddinov3_minimal/checkpoints/model.pth'
RESULTS_DIR   = '/content/drive/MyDrive/repa-results'

os.environ['MEDDINOV3_PATH'] = MEDDINOV3
os.environ['MEDDINOV3_CKPT'] = CKPT_PATH

os.makedirs(DATA_DIR, exist_ok=True)

print(f'MedDINOv3 exists  : {os.path.isdir(MEDDINOV3)}')
print(f'Checkpoint exists : {os.path.isfile(CKPT_PATH)}')
print(f'CheXpert exists   : {os.path.isdir(CHEXPERT_ROOT)}')
print(f'train.csv exists  : {os.path.isfile(os.path.join(CHEXPERT_ROOT, "train.csv"))}')

## 6. Prepare CheXpert
Reads chest X-ray images, converts to RGB, resizes to 256×256, MedVAE-encodes, saves latents.

Binary labels: **0 = normal** ("No Finding"), **1 = abnormal** (any pathology).

In [ ]:
import subprocess, sys

cmd = [
    sys.executable, '/content/REPA/prepare_chexpert.py',
    '--out-dir',       '/content/data/chexpert_256',
    '--chexpert-root', '/content/chexpert',
    '--resolution',    '256',
    '--max-samples',   '5000',
    '--vae-type',      'medvae',
]
result = subprocess.run(cmd, text=True)
print('Exit code:', result.returncode)

## 7. Train
Runs 15000 steps. Checkpoints saved to Google Drive.

**Models**
1. SiT-S/8 (Diffusion Model): denoises in latent space
2. MedDINOv3 ViT-Base: pretrained medical encoder
3. MedVAE (8×, 4ch): medical diffusion VAE

**Losses**
1. denoising_loss: standard diffusion loss
2. proj_loss: REPA alignment loss

Total loss = denoising_loss + proj_loss × proj_coeff

In [ ]:
import subprocess, os, re
from tqdm.notebook import tqdm

env = os.environ.copy()
env['MEDDINOV3_PATH'] = '/content/meddinov3/meddinov3_minimal'
env['MEDDINOV3_CKPT'] = '/content/meddinov3/meddinov3_minimal/checkpoints/model.pth'

cmd = [
    'accelerate', 'launch',
    '--mixed_precision', 'fp16',
    '--num_processes', '1',
    '/content/REPA/train.py',
    '--exp-name',            'repa-sits8-meddinov3-chexpert',
    '--output-dir',          '/content/drive/MyDrive/repa-results',
    '--report-to',           'none',
    '--model',               'SiT-S/8',
    '--num-classes',         '2',
    '--encoder-depth',       '8',
    '--enc-type',            'meddinov3-vit-b',
    '--data-dir',            '/content/data/chexpert_256',
    '--resolution',          '256',
    '--batch-size',          '16',
    '--num-workers',         '2',
    '--epochs',              '200',
    '--learning-rate',       '1e-4',
    '--mixed-precision',     'fp16',
    '--proj-coeff',          '0.5',
    '--cfg-prob',            '0.1',
    '--path-type',           'linear',
    '--max-train-steps',     '15000',
    '--checkpointing-steps', '15000',
    '--sampling-steps',      '99999999',
    '--vae-type',            'medvae',
]

process = subprocess.Popen(cmd, cwd='/content/REPA', env=env,
                           stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)

last_step = 0
pbar = tqdm(total=15000, desc='Training')
for line in process.stdout:
    m = re.search(r'Steps:\s*(\d+)/\d+.*?loss=([\d.]+).*?proj_loss=(-?[\d.]+)', line)
    if m:
        step = int(m.group(1))
        if step > last_step:
            pbar.update(step - last_step)
            pbar.set_postfix({'loss': m.group(2), 'proj': m.group(3)})
            last_step = step
pbar.close()
process.wait()
print('Training complete. Checkpoint saved.')

## 8. Check checkpoint

In [ ]:
import glob
print(glob.glob('/content/drive/MyDrive/repa-results/**/*.pt', recursive=True))

## 9. Generate samples

In [ ]:
import subprocess, os, glob

env = os.environ.copy()
env['MEDDINOV3_PATH'] = '/content/meddinov3/meddinov3_minimal'
env['MEDDINOV3_CKPT'] = '/content/meddinov3/meddinov3_minimal/checkpoints/model.pth'

ckpts = sorted(glob.glob('/content/drive/MyDrive/repa-results/repa-sits8-meddinov3-chexpert/checkpoints/*.pt'))
print('Using checkpoint:', ckpts[-1])

cmd = [
    'torchrun', '--nproc_per_node=1',
    '/content/REPA/generate.py',
    '--model',                'SiT-S/8',
    '--ckpt',                 ckpts[-1],
    '--encoder-depth',        '8',
    '--num-classes',          '2',
    '--projector-embed-dims', '768',
    '--per-proc-batch-size',  '16',
    '--num-fid-samples',      '2000',
    '--path-type',            'linear',
    '--mode',                 'ode',
    '--num-steps',            '50',
    '--cfg-scale',            '1.0',
    '--resolution',           '256',
    '--vae',                  'medvae',
]

result = subprocess.run(cmd, cwd='/content/REPA', env=env, text=True)
print('Exit code:', result.returncode)

## 10. Show generated images

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import glob

npz_files = sorted(glob.glob('/content/REPA/samples/**/*.npz', recursive=True))

data = np.load(npz_files[-1])
imgs = data[data.files[0]]

fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for i, ax in enumerate(axes.flat):
    ax.imshow(imgs[i].astype('uint8'), cmap='gray')
    ax.axis('off')
plt.tight_layout()
plt.show()
print(f'From: {npz_files[-1]}')

## 11. Check FID Score

In [ ]:
import subprocess, glob, numpy as np, os
from PIL import Image
from tqdm import tqdm

subprocess.run(['pip', 'install', '-q', 'torch-fidelity'], check=True)

gen_npz = sorted(glob.glob('/content/REPA/samples/**/*.npz', recursive=True))[-1]
print('Generated:', gen_npz)

gen_dir = '/content/gen_images'
os.makedirs(gen_dir, exist_ok=True)
data = np.load(gen_npz)
imgs = data[data.files[0]]
for i, img in enumerate(tqdm(imgs, desc='Saving generated images')):
    Image.fromarray(img.astype('uint8')).save(f'{gen_dir}/{i:05d}.png')

real_dir = '/content/data/chexpert_256/images/train'

subprocess.run([
    'fidelity',
    '--gpu', '0',
    '--fid',
    '--input1', gen_dir,
    '--input2', real_dir,
], text=True)